# Vector Database & RAG — Quick Revision Notes

## 1. What an Embedding Model Does

An embedding model converts text into a high-dimensional vector (list of numbers).

```text
"The capital of France is Paris"
        ↓
Embedding Model
        ↓
[0.12, -0.44, 0.91, ...]
```

The vector captures the semantic meaning of the text.

---

## 2. What a Vector Database Stores

A common misconception is that a vector DB stores only vectors.

In reality, it typically stores:

```python
{
    "text": "The capital of France is Paris",
    "embedding": [0.12, -0.44, 0.91, ...],
    "metadata": {
        "source": "document.pdf",
        "page": 5
    }
}
```

### Mental Model

| Text Chunk               | Embedding Vector   | Metadata |
| ------------------------ | ------------------ | -------- |
| France capital is Paris  | [0.12, -0.44, ...] | page=5   |
| Eiffel Tower is in Paris | [0.81, 0.24, ...]  | page=6   |
| India capital is Delhi   | [0.34, -0.12, ...] | page=7   |

---

## 3. What `Chroma.from_documents()` Does

When you run:

```python
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings
)
```

Conceptually:

```text
For each chunk:
    Generate embedding
    Store:
        - Original text
        - Embedding vector
        - Metadata
```

---

## 4. Retrieval Process

### User Query

```text
"What is the capital of France?"
```

### Step 1: Query Embedding

```text
Query
  ↓
Embedding Model
  ↓
Query Vector
```

Example:

```python
[0.11, -0.41, 0.87, ...]
```

---

### Step 2: Similarity Search

Compare query vector against stored vectors:

```python
similarity(query_vector, vector_A)
similarity(query_vector, vector_B)
similarity(query_vector, vector_C)
```

Common similarity metrics:

* Cosine Similarity
* Dot Product
* Euclidean Distance

Example:

```text
A → 0.98
B → 0.72
C → 0.41
```

Chunk A is selected as most relevant.

---

## 5. Most Important Concept

### Vector DB Does NOT Convert Vectors Back Into Text

There is no such process:

```text
Vector
 ↓
Reverse Embedding
 ↓
Text
```

This does not exist.

---

### What Actually Happens

The original text was already stored:

```python
{
    "text": "The capital of France is Paris",
    "embedding": [...]
}
```

After similarity search:

```text
Matching Vector Found
        ↓
Return Associated Text
```

The text is retrieved directly from storage.

---

## 6. What LangChain Retriever Returns

When you run:

```python
retriever.invoke(query)
```

You get:

```python
[
    Document(
        page_content="The capital of France is Paris",
        metadata={...}
    )
]
```

You do NOT get vectors back.

LangChain uses vectors internally and returns the matching text chunks.

---

## 7. Complete RAG Pipeline

```text
PDF / Documents
        ↓
Chunking
        ↓
Embedding Model
        ↓
Vectors
        ↓
Vector Database
(Text + Vector + Metadata Stored)

================================

User Query
        ↓
Embedding Model
        ↓
Query Vector
        ↓
Similarity Search
        ↓
Top Matching Chunks
        ↓
Retriever Returns Documents
        ↓
LLM
        ↓
Final Answer
```

---

## One-Line Revision

**A vector database stores the original text along with its embedding vector. During retrieval, the query is embedded, similarity search finds the closest vectors, and the database returns the associated stored text—not a reconstructed version of the text.**


##### Different embedding models usually produce different vector spaces because they may differ in training data, architecture, initialization, optimization, and learned weights. Even if two models are trained on the same data, their vectors can still be different. Similarity search works only when document embeddings and query embeddings are generated from the same embedding space (typically the same model/checkpoint).